In [1]:
# ライブラリーのインポート
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models

In [2]:
#データフォルダのパス
base_dir = "dog_cat_photos"
train_dir = base_dir + "/train"
test_dir = base_dir + "/test"
#画像サイズ
img_size = (224,224)
batch_size = 32

In [4]:
# 学習用(データ水増しあり)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
# テスト用(データ水増しなし)
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [6]:
# 画像データの読み込み
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary"
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle=False
)

Found 300 images belonging to 2 classes.
Found 100 images belonging to 2 classes.


In [10]:
# 転移学習用のデータモデルの読み込み
base_model = MobileNetV2(
    weights=None,
    include_top=False,
    input_shape=(224, 224,3)
)

# 特徴抽出のみ使用
base_model.trainable = False

In [12]:
# モデルの構築
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

In [13]:
# コンパイル
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [14]:
# 学習の実行
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 431ms/step - accuracy: 0.5000 - loss: 0.6937 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 393ms/step - accuracy: 0.5000 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 352ms/step - accuracy: 0.5000 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 372ms/step - accuracy: 0.5000 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 342ms/step - accuracy: 0.5000 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 368ms/step - accuracy: 0.5000 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 349ms/step - accuracy: 0.5000 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 355ms/step - accuracy: 0.5000 - loss: 0.6932 - val_accuracy: 0.

In [15]:
loss, acc = model.evaluate(test_generator)
print("テストデータでの正答率：", acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 237ms/step - accuracy: 0.5000 - loss: 0.6932
テストデータでの正答率： 0.5
